## TP3

Dans ce TP, nous nous intéressons aux méthodes ensemblistes et comment en instancier avec sk-learn.
On se restreint maintenant au jeu de données _digits_.


### Bagging 

Utilisons le module [sklearn.ensemble.BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html) pour instancier des méthodes de bagging.

**§ Rappeler en quoi consiste le bagging**


La bagging consiste à crée plusieurs sous eu de données avec remise que l'on entraine souvent via des arbre de décision. On combine les résultats soit via vote pour la classification soit via moyenne des pondérations pour la régrésion.


**§ Regarder la doc et expliquer les six premiers paramètres listés et le choix des valeurs par défaut**


1. `estimator=None` &rarr; C’est le modèle de base qui sera dupliqué et entraîné plusieurs fois. Par default le modèle est `DecisionTreeClassifier`
2. `n_estimators=10` &rarr; C’est le nombre de modèles entraînés dans l’ensemble. 10 par defaut qui est un compromis entre coût de calcul et performance. 
3. `max_samples=None` &rarr; C’est le nombre d’observations tirées pour entraîner chacun des modèles. Par default on tire autant de fois que l'on a d'observation. 
4. `max_features=1.0` &rarr; Même principe que max_samples, mais cette fois-ci pour les variables / features. Par défault 1 car la bagging randomise principalement les observations pas les feature. 
5. `bootstrap=True` &rarr; Indique si le tirage est avec remise. Par default oui car c'est le principe même du bagging. 
6. `bootstrap_features=False` &rarr; Même principe mais pour les features. False par default car initialement on fait ça sur les observations. 


**§ A l'aide des paramètres booléens, définir 4 versions, les comparer et analyser les résultats** 

In [ ]:
import sklearn
from sklearn.datasets import load_digits
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import train_test_split

RANDOM_STATE = 2
SIZE = 6

CONFIG = {
    "Version 1": {
        "bootstrap": True,
        "bootstrap_features": False
    },
    "Version 2": {
        "bootstrap": False,
        "bootstrap_features": False
    },
    "Version 3": {
        "bootstrap": True,
        "bootstrap_features": True
    },
    "Version 4": {
        "bootstrap": False,
        "bootstrap_features": True
    }
}

j_digits = load_digits()
X, Y = j_digits.data, j_digits.target

results = {}

for name, value in CONFIG.items():
    score = []

    for seed in range(1,31):
        X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.33, random_state=seed)
        bg = BaggingClassifier(bootstrap=value['bootstrap'], bootstrap_features=value['bootstrap_features'], random_state=seed).fit(X_train, Y_train)
        score.append(bg.score(X_test, Y_test))
    
    results[name] = score
    score = []


In [8]:
def sum(lst):
    res = 0
    for e in lst:
        res += e
    return res

print(f"Version 1 : {sum(results['Version 1']) / len(results['Version 1'])}")
print(f"Version 2 : {sum(results['Version 2']) / len(results['Version 2'])}")
print(f"Version 3 : {sum(results['Version 3']) / len(results['Version 3'])}")
print(f"Version 4 : {sum(results['Version 4']) / len(results['Version 4'])}")

Version 1 : 0.922783389450056
Version 2 : 0.8546576879910209
Version 3 : 0.9372615039281708
Version 4 : 0.9383838383838383


### Random Forest

Utilisons le module [sklearn.ensemble.RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) pour instancier des random forest. Rappelons que les random forest sont issues des techniques de bagging appliquées aux arbres de décisions.

**§ Regarder la doc et identifier les paramètres qui permettent notamment de définir le nombre d'arbres et leur profondeur maximale**


- `n_estimators=100` &rarr; Nombre d'arbre dans la forêts.
- `max_depth=None` &rarr; Profondeur max de chaque arbre. 


En regardant et comparant les paramètres de ce module avec ceux du module de bagging précédent, il est clair qu'il n'est pas possible de répliquer toutes les techniques de bagging précédentes et notamment concernant les attributs.

**§ Quelle(s) version(s) de bagging vu(es) précédemment n'est-il pas possible d'utiliser avec ce module ?**


C'est le `bootstrap_features`.


**§ Proposer une instanciation utilisant le module de bagging pour pouvoir lever cette impossibilité**


On utilise `BaggingClassifier` avec `DecisionTreeClassifier`. Le `n_estimators` devient donc le nombre d'arbre que l'on veut. 



L'objectif des expérimentations suivantes est de vérifier le concept "d'apprenants faibles" qui sont "forts ensemble". Pour cela, vous devez analyser l'impact du nombre d'arbres et de la profondeur max des arbres sur la performance et le temps d'exécution.

**§ Rédiger un protocole expérimental, exécuter puis analyser les résultats**


On va faire varier la profondeur maximal, et le nombre d'arbre. Et pour chaque on va le faire pour un random_state de 1 à 30 où l'on prendre sa moyenne.

On va tester pour cela : 
- Les profondeurs 1 - 2 - 4 - 8 et None 
- Les nombres nombre d'arbres 5 - 15 - 40 et 100 

Il faudra donc les faire varier en même temps. 

Dans un premier temps on comparera les accurancys moyennes pour voir rapidement les soucis. Ensuite on mettra des tests statistiques pour s'assurer de qui sont les meilleurs modèles.  
Pour cela on mettra en place un test de Friedman pour vérifier si il y a bien des différences statistiques entre des modèles et si oui le test de Wilcoxon appariés

In [13]:
from sklearn.tree import DecisionTreeClassifier

def test_bagging(random_state, max_depth, n_estimators):
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.33, random_state=random_state)
    bg = BaggingClassifier(
    random_state=random_state,
    estimator=DecisionTreeClassifier(
        max_depth=max_depth
    ), 
    n_estimators=n_estimators
    ).fit(X_train, Y_train)

    return bg.score(X_test, Y_test)

DEPTHS = [1, 2, 4, 8, None]
N_ESTIMATORS = [5, 15, 40, 100]

results = []

for max_depth in DEPTHS:
    for estimators in N_ESTIMATORS:
        score = []
        for seed in range(1, 31):
            score.append(test_bagging(seed, max_depth, estimators))
        results.append({
            "depth" : max_depth,
            "estimators": estimators,
            "accurancy": sum(score) / len(score),
            "results": score
        })
        score = []        


In [19]:
score = []
for seed in range(1, 31):
    score.append(test_bagging(seed, 3, 1000))
print(sum(score) / len(score))

0.6916947250280583


In [17]:
score = []
for seed in range(1, 31):
    score.append(test_bagging(seed, 10, 30))
print(sum(score) / len(score))

0.9363636363636362


In [14]:
for result in results: 
    print(result)

{'depth': 1, 'estimators': 5, 'accurancy': 0.2877665544332212, 'results': [0.27104377104377103, 0.3872053872053872, 0.2777777777777778, 0.42424242424242425, 0.28619528619528617, 0.2946127946127946, 0.3383838383838384, 0.2558922558922559, 0.25925925925925924, 0.27441077441077444, 0.3956228956228956, 0.26430976430976433, 0.26262626262626265, 0.23905723905723905, 0.16666666666666666, 0.3872053872053872, 0.26430976430976433, 0.2895622895622896, 0.23905723905723905, 0.2441077441077441, 0.1717171717171717, 0.36363636363636365, 0.32996632996632996, 0.3265993265993266, 0.3400673400673401, 0.24915824915824916, 0.17676767676767677, 0.3265993265993266, 0.265993265993266, 0.2609427609427609]}
{'depth': 1, 'estimators': 15, 'accurancy': 0.3299102132435465, 'results': [0.40235690235690236, 0.4158249158249158, 0.2777777777777778, 0.4393939393939394, 0.4225589225589226, 0.3787878787878788, 0.4074074074074074, 0.30134680134680136, 0.25925925925925924, 0.27441077441077444, 0.39057239057239057, 0.2643097

In [15]:
from scipy.stats import friedmanchisquare

scores = [
    result["results"]
    for result in results
]

stat, p_value = friedmanchisquare(*scores)

print("Statistique de Friedman :", stat)
print("p-value :", p_value)

if p_value < 0.05:
    print("Il existe une différence significative entre les modèles.")
else:
    print("Aucune différence significative détectée.")

Statistique de Friedman : 559.3420913950159
p-value : 1.877817041772597e-106
Il existe une différence significative entre les modèles.


### Boosting

Plusieurs techniques de boosting sont possibles. 
Nous regarderons ici ADABoost avec le module [sklearn.ensemble.AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html),
le Gradient boosting avec le module [sklearn.ensemble.GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html)
et une variante plus efficace [xgboost](https://xgboost.readthedocs.io/en/latest/parameter.html).

XGBoost est une bibliothèque, hors sklearn. Pour l'installer puis l'importer, taper les commandes suivantes dans des cellules différentes :
```
pip install xgboost
pip install --upgrade xgboost
from xgboost import XGBClassifier
```

**§ Regarder la doc et indiquer précisément la méthode de classification utilisée**


Les trois méthodes sont des méthodes de boosting basées sur des arbres de décision. AdaBoost ré-entraîne des arbres en donnant davantage de poids aux observations mal classées. Gradient Boosting construit des arbres qui corrigent progressivement le gradient de la fonction de perte. XGBoost reprend le principe du Gradient Boosting avec une optimisation plus poussée, utilisant notamment les dérivées d'ordre 1 et 2 de la fonction de perte.

AdaBoostClassifier utilise par défaut des arbres de décision très faibles, appelés decision stumps : ce sont des DecisionTreeClassifier(max_depth=1). À chaque itération, AdaBoost augmente le poids des observations mal classées pour que l’arbre suivant se concentre davantage dessus. Sur un problème multiclasse comme digits, sklearn utilise l'algorithme SAMME.

GradientBoostingClassifier construit également des arbres successivement, mais le principe est différent : chaque nouvel arbre cherche à corriger le gradient de l’erreur du modèle précédent. Pour la classification, la fonction de coût par défaut est la log_loss. En multiclasse, sklearn entraîne un arbre de régression par classe et par étape de boosting.

XGBClassifier est une version optimisée du Gradient Boosting. Il utilise également des arbres successifs, mais exploite notamment gradient et Hessienne de la fonction de coût, ainsi que diverses régularisations et optimisations. Pour une classification multiclasse, l'objectif approprié est multi:softprob.


**§ Instancier 3 méthodes de boosting différentes mais comparable, les exécuter et analyser les résultats**

In [22]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score


def test_boosting(random_state):

    X_train, X_test, Y_train, Y_test = train_test_split(
        X,
        Y,
        test_size=0.33,
        random_state=random_state,
        stratify=Y
    )

    # AdaBoost
    ada = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=100,
        learning_rate=0.1,
        random_state=random_state
    )

    # Gradient Boosting
    gradient = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=1,
        random_state=random_state
    )

    # XGBoost
    xgb = XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=1,
        objective="multi:softprob",
        random_state=random_state
    )

    ada.fit(X_train, Y_train)
    gradient.fit(X_train, Y_train)
    xgb.fit(X_train, Y_train)

    acc_ada = accuracy_score(Y_test, ada.predict(X_test))
    acc_gradient = accuracy_score(Y_test, gradient.predict(X_test))
    acc_xgb = accuracy_score(Y_test, xgb.predict(X_test))

    return acc_ada, acc_gradient, acc_xgb

results_ada = []
results_gradient = []
results_xgb = []

for random_state in range(1, 31):

    acc_ada, acc_gradient, acc_xgb = test_boosting(random_state)

    results_ada.append(acc_ada)
    results_gradient.append(acc_gradient)
    results_xgb.append(acc_xgb)

import numpy as np

print("AdaBoost :", np.mean(results_ada))
print("Gradient Boosting :", np.mean(results_gradient))
print("XGBoost :", np.mean(results_xgb))

AdaBoost : 0.6068462401795737
Gradient Boosting : 0.9333333333333333
XGBoost : 0.9125701459034795
